In [1]:
from datasets import load_dataset
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from transformers import (
    BertTokenizer,
    BertModel
)
from torch.nn.functional import pad
import torch.nn as nn
from tqdm import tqdm

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

import pandas as pd

import torch

print(torch.cuda.is_available())


/home/dominik/miniconda3/envs/hf-sentiment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True


https://www.kaggle.com/datasets/rmisra/news-category-dataset?utm_source=chatgpt.com


In [40]:
with open("data/data.json", "w") as f1:
    f1.write("[")
    with open("data/News_Category_Dataset_v3.json", "r") as f2:
        for i, line in enumerate(f2.readlines()):
            line = line.strip()
            line = line + ","
            f1.write(line + "\n")
    f1.write("]")
# Delete the last comma from file manually.

In [2]:
import json

with open("data/data.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

In [3]:
df = pd.DataFrame(dataset)
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [4]:
X_train = list(df["short_description"])
y_train = list(df["category"])
set(y_train)

{'ARTS',
 'ARTS & CULTURE',
 'BLACK VOICES',
 'BUSINESS',
 'COLLEGE',
 'COMEDY',
 'CRIME',
 'CULTURE & ARTS',
 'DIVORCE',
 'EDUCATION',
 'ENTERTAINMENT',
 'ENVIRONMENT',
 'FIFTY',
 'FOOD & DRINK',
 'GOOD NEWS',
 'GREEN',
 'HEALTHY LIVING',
 'HOME & LIVING',
 'IMPACT',
 'LATINO VOICES',
 'MEDIA',
 'MONEY',
 'PARENTING',
 'PARENTS',
 'POLITICS',
 'QUEER VOICES',
 'RELIGION',
 'SCIENCE',
 'SPORTS',
 'STYLE',
 'STYLE & BEAUTY',
 'TASTE',
 'TECH',
 'THE WORLDPOST',
 'TRAVEL',
 'U.S. NEWS',
 'WEDDINGS',
 'WEIRD NEWS',
 'WELLNESS',
 'WOMEN',
 'WORLD NEWS',
 'WORLDPOST'}

We have very imbalanced dataset

In [5]:
def proportions(df):
    N = len(df)
    xs = df.groupby("category").size() / N * 100
    print(xs.sum())
    print(xs)

proportions(df)

100.0
category
ARTS               0.720194
ARTS & CULTURE     0.639058
BLACK VOICES       2.187308
BUSINESS           2.859775
COLLEGE            0.545992
COMEDY             2.577233
CRIME              1.700020
CULTURE & ARTS     0.512583
DIVORCE            1.635111
EDUCATION          0.483947
ENTERTAINMENT      8.286283
ENVIRONMENT        0.689171
FIFTY              0.668649
FOOD & DRINK       3.025863
GOOD NEWS          0.667217
GREEN              1.251390
HEALTHY LIVING     3.194815
HOME & LIVING      2.061787
IMPACT             1.662793
LATINO VOICES      0.539310
MEDIA              1.405070
MONEY              0.838078
PARENTING          4.195641
PARENTS            1.887585
POLITICS          16.991605
QUEER VOICES       3.029204
RELIGION           1.229913
SCIENCE            1.052848
SPORTS             2.423077
STYLE              1.075756
STYLE & BEAUTY     4.683883
TASTE              1.000348
TECH               1.004167
THE WORLDPOST      1.748701
TRAVEL             4.724928
U.S. 

In [94]:
df["short_description"][1122]

'“I can’t put a date or exact time on it, but everything is in place for Russia to move forward," the secretary of state said.'

In [6]:
label_map = {
    "ARTS": 0,
    "ARTS & CULTURE": 1,
    "BLACK VOICES": 2,
    "BUSINESS": 3,
    "COLLEGE": 4,
    "COMEDY": 5,
    "CRIME": 6,
    "CULTURE & ARTS": 7,
    "DIVORCE": 8,
    "EDUCATION": 9,
    "ENTERTAINMENT": 10,
    "ENVIRONMENT": 11,
    "FIFTY": 12,
    "FOOD & DRINK": 13,
    "GOOD NEWS": 14,
    "GREEN": 15,
    "HEALTHY LIVING": 16,
    "HOME & LIVING": 17,
    "IMPACT": 18,
    "LATINO VOICES": 19,
    "MEDIA": 20,
    "MONEY": 21,
    "PARENTING": 22,
    "PARENTS": 23,
    "POLITICS": 24,
    "QUEER VOICES": 25,
    "RELIGION": 26,
    "SCIENCE": 27,
    "SPORTS": 28,
    "STYLE": 29,
    "STYLE & BEAUTY": 30,
    "TASTE": 31,
    "TECH": 32,
    "THE WORLDPOST": 33,
    "TRAVEL": 34,
    "U.S. NEWS": 35,
    "WEDDINGS": 36,
    "WEIRD NEWS": 37,
    "WELLNESS": 38,
    "WOMEN": 39,
    "WORLD NEWS": 40,
    "WORLDPOST": 41
}

df["label"] = df["category"].map(lambda x : label_map[x])
df.head()

,link,headline,category,short_description,authors,date,label
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23,35
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23,35
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23,5
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23,22
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22,35


In [7]:
class NewsDataset(Dataset):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    
    def __init__(self, df):
        self.X_data = list(df["short_description"])
        self.y_data = list(df["label"])

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):
        tokens = self.tokenizer(self.X_data[idx], return_tensors="pt")
        return tokens.input_ids, torch.tensor(self.y_data[idx])

db = NewsDataset(df)
X,y = db[0]
print(X, y)
print(len(db))


tensor([[  101,  2740,  8519,  2056,  2009,  2003,  2205,  2220,  2000, 16014,
          3251,  5157,  2052,  2674,  2039,  2007,  1996, 18225,  2454, 21656,
          1997,  1996,  2047, 23715,  2015,  1996,  1057,  1012,  1055,  1012,
          3641,  2005,  1996,  2991,  1012,   102]]) tensor(35)
209527


We want to stratify the dataset and preserve the class distribution from original dataset. We want to preserve the distibution in train and test set. We print the class proportion for the train and test datasets and observe that they are apporiximately equal. 

In [8]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)
db_train = NewsDataset(train_df)
db_test  = NewsDataset(test_df)

proportions(train_df)
proportions(test_df)

99.99999999999999
category
ARTS               0.720077
ARTS & CULTURE     0.638941
BLACK VOICES       2.187077
BUSINESS           2.860024
COLLEGE            0.545874
COMEDY             2.577243
CRIME              1.700264
CULTURE & ARTS     0.512466
DIVORCE            1.635237
EDUCATION          0.483830
ENTERTAINMENT      8.285955
ENVIRONMENT        0.689054
FIFTY              0.668771
FOOD & DRINK       3.025874
GOOD NEWS          0.666981
GREEN              1.251633
HEALTHY LIVING     3.194707
HOME & LIVING      2.061794
IMPACT             1.662679
LATINO VOICES      0.539312
MEDIA              1.404955
MONEY              0.838200
PARENTING          4.195775
PARENTS            1.887592
POLITICS          16.991308
QUEER VOICES       3.029453
RELIGION           1.230156
SCIENCE            1.052971
SPORTS             2.423324
STYLE              1.075641
STYLE & BEAUTY     4.683781
TASTE              1.000471
TECH               1.004051
THE WORLDPOST      1.748588
TRAVEL             4.

In [9]:
def merge_batch(xs):
    inputs_shape = [x[0].shape[1] for x in xs] 
    m = max(inputs_shape)
    pad_nums = [m - x[0].shape[1] for x in xs]
    input_tokens = [pad(x[0], (0, pad_nums[i])) for i, x in enumerate(xs)]
    targets = [x[1] for x in xs]
    input_batch = torch.stack(input_tokens)
    attention_mask = [torch.cat((torch.ones(x), torch.zeros(pad_nums[i])), dim=0) for i, x in enumerate(inputs_shape)]
    attention_mask = torch.stack(attention_mask)
    return input_batch.squeeze(1), attention_mask, torch.tensor(targets)


train_loader = DataLoader(
    db_train,
    batch_size=3,
    shuffle=True,
    collate_fn=merge_batch
)

test_loader = DataLoader(
    db_test,
    batch_size=16,
    shuffle=False
)

input, att, out = next(iter(train_loader))
print(input.shape, att.shape, out.shape)



torch.Size([3, 47]) torch.Size([3, 47]) torch.Size([3])


In [14]:
class Classifier(nn.Module):
    model_name = "bert-base-uncased"

    def __init__(self, num_classes, input_size=768):
        super().__init__()
        self.model = BertModel.from_pretrained(self.model_name)
        self.classifier = nn.Linear(
            input_size,
            num_classes
        )

    def forward(self, x, attention_mask):
        hs = self.model(input_ids=x, 
                   attention_mask=attention_mask).last_hidden_state

        cls_embedding = hs[:,0,:]
        return self.classifier(cls_embedding)


cls = Classifier(len(label_map))
cls.classifier


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7296.16it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Linear(in_features=768, out_features=42, bias=True)

In [15]:
train_loader = DataLoader(
    db_train,
    batch_size=18,
    shuffle=True,
    collate_fn=merge_batch
)

test_loader = DataLoader(
    db_test,
    batch_size=18,
    shuffle=True,
    collate_fn=merge_batch
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Classifier(len(label_map))
model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5350.94it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
def evaluate(model, test_loader):
    device = "cuda"
    model = model.to(device)
    model.eval()
    all_predictions = []
    all_targets     = [] 
    with torch.no_grad():
        total_loss = 0
        correct = 0
        total = 0
        for inputs, attention_mask, targets in tqdm(test_loader, desc="Test"):
            inputs = inputs.to(device)
            attention_mask = attention_mask.to(device)
            targets = targets.to(device)
            logits = model(inputs, attention_mask)
            loss = criterion(logits, targets)

            # Statistics
            total_loss += loss.item()
            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)

            # Move to CPU
            all_predictions.extend(predictions.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    precision = precision_score(
        all_targets,
        all_predictions,
        average="macro"
    )

    recall = recall_score(
        all_targets,
        all_predictions,
        average="macro"
    )

    f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro"
    )
    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(f"Eval Loss: {avg_loss:.4f}" 
          f" Accuracy: {accuracy:.4f}" 
          f" Recall: {recall:.4f}" 
          f" Precision: {precision:.4f}"
          f" F score: {f1:.4f}")

evaluate(model, test_loader)

Test: 100%|██████████| 2329/2329 [02:34<00:00, 15.04it/s]

Eval Loss: 0.9487 Accuracy: 0.0272 Recall: 0.0316 Precision: 0.0371 F score: 0.0114



/home/dominik/miniconda3/envs/hf-sentiment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [25]:
train_loader = DataLoader(
    db_train,
    batch_size=18,
    shuffle=True,
    collate_fn=merge_batch
)

test_loader = DataLoader(
    db_test,
    batch_size=18,
    shuffle=True,
    collate_fn=merge_batch
)

device = torch.device(
    "cpu" if torch.cuda.is_available() else "cuda"
)

model = Classifier(len(label_map))
model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

num_epochs=3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for inputs, attention_mask, targets in tqdm(train_loader, desc="Training"):
        inputs = inputs.to(device)
        attention_mask = attention_mask.to(device)
        targets = targets.to(device)
        # Clear gradients
        optimizer.zero_grad()
        # Forward pass
        logits = model(inputs, attention_mask)
        # Calculate loss
        loss = criterion(logits, targets)
        # Backpropagation
        loss.backward()
        # Update parameters
        optimizer.step()

        # Statistics
        total_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"Loss: {avg_loss:.4f} "
        f"Accuracy: {accuracy:.4f}"
    )
    evaluate(model, test_loader)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5112.94it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Training:   0%|          | 11/9313 [00:33<7:56:55,  3.08s/it]


KeyboardInterrupt: 